# Loan Approval — Complete Feature Engineering

### Teaching Notebook

This notebook follows the structure and teaching flow of the provided **Feature_Engineering.ipynb** and applies the concepts to the `loan_approved.csv` dataset.

**Flow**

1. Introduction to Feature Engineering
2. Understand the loan dataset
3. Data profiling with YData Profiling
4. Null / missing values
5. Outliers
6. Preprocessing
7. Feature creation
8. Encoding
9. Transformation
10. Scaling
11. Train/test split and data leakage
12. Final model-ready pipeline
13. Student practice checklist

> **Important:** Feature engineering is not about applying every technique at once. We inspect the data, choose an appropriate technique, and validate the result.

## 1. What is Feature Engineering?

Feature Engineering is the process of selecting, cleaning, transforming, encoding, scaling, and creating useful variables from raw data so that machine-learning algorithms can use the information effectively.

A simple story:

**Raw data → Clean data → Meaningful features → Model-ready data → Machine Learning model**

A feature is useful when it helps represent information relevant to the target.

For this loan problem, `Loan_ID` is an identifier, not a meaningful predictive feature, so it should not be given to the model.

## 2. Why do we need Feature Engineering?

Real-world data can contain:

- Missing values
- Outliers
- Different data types
- Categorical variables
- Skewed numerical variables
- Features measured on different scales
- Irrelevant columns
- Information that can be represented more meaningfully by combining existing variables

Feature engineering is therefore an iterative process: inspect → transform → validate → improve.

# 3. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import boxcox

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PowerTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

# 4. Load the Loan Dataset

In [ ]:
df = pd.read_csv("loan_approved.csv")

print("Shape:", df.shape)
df.head()

## 5. Understand the Dataset

Always inspect the dataset before choosing feature-engineering techniques.

In [ ]:
print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nStatistical summary:")
display(df.describe(include="all").T)

# 6. Target Variable

The target is:

`Loan_Status (Approved)`

- `Y` → Loan approved
- `N` → Loan not approved

We will eventually convert this target to 1 and 0.

In [ ]:
print(df["Loan_Status (Approved)"].value_counts())
print()
print(df["Loan_Status (Approved)"].value_counts(normalize=True))

# 7. YData Profiling

YData Profiling automatically creates an exploratory report containing information such as:

- Variables and data types
- Missing values
- Unique values
- Distributions
- Descriptive statistics
- Correlations
- Duplicates
- Warnings about potentially problematic variables

Install if required:

```python
!pip install ydata-profiling
```

In [ ]:
# Uncomment if the package is not installed:
# !pip install ydata-profiling

from ydata_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="Loan Approval Data Profiling Report",
    explorative=True
)

profile.to_notebook_iframe()

In [ ]:
# Save the report so it can be opened in a browser.
profile.to_file("loan_approval_profile.html")

# 8. Missing / Null Values

A missing value means that the value for a variable is unavailable.

First identify missing values. Never start filling values before understanding which columns are affected.

In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nMissing percentage:")
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
display(missing_pct)

## Useful Pandas functions for missing data

- `isnull()` / `isna()` → identify missing values
- `notnull()` / `notna()` → identify non-missing values
- `info()` → inspect types and non-null counts
- `dropna()` → remove rows/columns with missing values
- `fillna()` → replace missing values
- `replace()` → replace specific values
- `drop_duplicates()` → remove duplicate rows
- `unique()` → inspect unique values

### Teaching point

Do not automatically delete every row containing a missing value. The correct treatment depends on the variable, amount of missingness, and business meaning.

# 9. Duplicate Rows

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

# If duplicates are confirmed to be accidental, they can be removed:
# df = df.drop_duplicates().copy()

# 10. Remove the Identifier

`Loan_ID` is a unique identifier.

An identifier is useful for tracking records, but normally it is not a meaningful predictive feature.

Therefore, remove it from the model features.

We keep the original dataframe unchanged for teaching/inspection and create a modeling dataframe.

In [ ]:
model_df = df.copy()

model_df = model_df.drop(columns=["Loan_ID"])

print(model_df.columns.tolist())

# 11. Numerical vs Categorical Features

For this dataset:

### Numerical
- ApplicantIncome
- CoapplicantIncome
- LoanAmount
- Loan_Amount_Term
- Credit_History

### Categorical
- Gender
- Married
- Dependents
- Education
- Self_Employed
- Property_Area

The target is `Loan_Status (Approved)`.

In [ ]:
target = "Loan_Status (Approved)"

numeric_features = [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History"
]

categorical_features = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "Property_Area"
]

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

# 12. Outliers

An outlier is an observation that is unusually far from the rest of the observations.

For this dataset, inspect:

- ApplicantIncome
- CoapplicantIncome
- LoanAmount

### Important teaching point

An outlier is not automatically a bad data point.

A high-income applicant can be a genuine observation. Therefore, investigate before deleting.

In [ ]:
for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
    plt.figure(figsize=(7, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot — {col}")
    plt.show()

## IQR Method

The Interquartile Range is:

**IQR = Q3 − Q1**

Common rule:

- Lower boundary = Q1 − 1.5 × IQR
- Upper boundary = Q3 + 1.5 × IQR

Values outside these boundaries are flagged as potential outliers.

In [ ]:
def iqr_summary(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower": lower,
        "Upper": upper,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Percentage": float(mask.mean() * 100)
    }

outlier_table = pd.DataFrame({
    col: iqr_summary(df, col)
    for col in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]
}).T

display(outlier_table)

# 13. Outlier Handling Options

Depending on the data and business problem, common choices are:

1. Keep genuine extreme observations
2. Remove confirmed data errors
3. Cap/winsorize extreme values
4. Apply a transformation
5. Use a model/scaler less sensitive to extreme values

We will **not blindly delete loan records**.

In [ ]:
# Optional demonstration of winsorization.
# Use only after deciding that capping is appropriate for the business problem.

from scipy.stats.mstats import winsorize

# Example only — creates a separate series and does not overwrite the original.
winsorized_income = winsorize(
    df["ApplicantIncome"],
    limits=[0.05, 0.05]
)

display(pd.Series(winsorized_income, name="ApplicantIncome_Winsorized").describe())

# 14. Feature Creation

Feature creation means deriving a new variable from existing variables.

This is where domain knowledge becomes important.

For loan approval, two especially useful pieces of information are:

- Applicant income
- Co-applicant income

Together they represent total household/application income.

In [ ]:
feature_df = model_df.copy()

feature_df["TotalIncome"] = (
    feature_df["ApplicantIncome"] +
    feature_df["CoapplicantIncome"]
)

display(
    feature_df[
        ["ApplicantIncome", "CoapplicantIncome", "TotalIncome"]
    ].head()
)

## 15. Convert Dependents

`Dependents` contains values such as `0`, `1`, `2`, and `3+`.

For a simple ordinal representation, we can map `3+` to `3`.

This preserves the ordered nature of the variable without dropping it.

In [ ]:
feature_df["Dependents"] = feature_df["Dependents"].replace({
    "3+": 3
})

# Keep missing values as missing for now.
feature_df["Dependents"] = pd.to_numeric(
    feature_df["Dependents"],
    errors="coerce"
)

display(feature_df["Dependents"].value_counts(dropna=False).sort_index())

# 16. Income per Dependent

We can create:

**IncomePerDependent = TotalIncome / (Dependents + 1)**

Why `+1`?

If an applicant has zero dependents, division by zero must be avoided.

This is a domain-inspired feature, not a universally correct feature. Its usefulness should be validated using the model.

In [ ]:
feature_df["IncomePerDependent"] = (
    feature_df["TotalIncome"] /
    (feature_df["Dependents"] + 1)
)

display(
    feature_df[
        ["TotalIncome", "Dependents", "IncomePerDependent"]
    ].head()
)

# 17. Loan-to-Income Ratio

A useful domain feature is the relationship between loan amount and total income.

**LoanIncomeRatio = LoanAmount / TotalIncome**

A larger value means the requested loan is larger relative to the applicant's available income.

In [ ]:
feature_df["LoanIncomeRatio"] = (
    feature_df["LoanAmount"] /
    feature_df["TotalIncome"]
)

display(
    feature_df[
        ["LoanAmount", "TotalIncome", "LoanIncomeRatio"]
    ].head()
)

## Avoid division-by-zero problems

If a denominator can be zero, use a safe calculation.

For this dataset, the income columns can contain zero values, so use `np.nan` where a ratio cannot be meaningfully calculated.

In [ ]:
feature_df["LoanToApplicantIncome"] = np.where(
    feature_df["ApplicantIncome"] > 0,
    feature_df["LoanAmount"] / feature_df["ApplicantIncome"],
    np.nan
)

display(
    feature_df[
        ["LoanAmount", "ApplicantIncome", "LoanToApplicantIncome"]
    ].head()
)

# 18. Encoding Categorical Variables

Machine-learning algorithms generally need numerical input.

### Common encoding approaches

- Binary mapping for two-category variables
- Ordinal encoding when categories have meaningful order
- One-hot encoding for nominal categories

### Loan dataset examples

Binary:
- Gender
- Married
- Education
- Self_Employed

Ordinal:
- Dependents (`0`, `1`, `2`, `3+`)

Nominal:
- Property_Area (`Urban`, `Semiurban`, `Rural`)

## Binary Encoding — Example

For a binary variable:

`Yes → 1`

`No → 0`

This is appropriate when there are exactly two categories and the numeric representation is simply a coding of the two states.

In [ ]:
binary_map = {
    "Gender": {"Male": 1, "Female": 0},
    "Married": {"Yes": 1, "No": 0},
    "Education": {"Graduate": 1, "Not Graduate": 0},
    "Self_Employed": {"Yes": 1, "No": 0}
}

encoded_demo = feature_df.copy()

for col, mapping in binary_map.items():
    encoded_demo[col] = encoded_demo[col].map(mapping)

display(encoded_demo[[
    "Gender", "Married", "Education", "Self_Employed"
]].head())

# 19. One-Hot Encoding

`Property_Area` has three nominal categories:

- Rural
- Semiurban
- Urban

There is no reason to assume:

`Urban > Semiurban > Rural`

Therefore one-hot encoding is appropriate.

Example:

| Property_Area | Rural | Semiurban | Urban |
|---|---:|---:|---:|
| Rural | 1 | 0 | 0 |
| Semiurban | 0 | 1 | 0 |
| Urban | 0 | 0 | 1 |

In a regression-style design matrix, one category can be dropped to avoid perfect redundancy.

In [ ]:
one_hot_demo = pd.get_dummies(
    feature_df,
    columns=["Property_Area"],
    drop_first=True,
    dtype=int
)

display(one_hot_demo.head())

# 20. Target Encoding

The target is:

- `Y` → 1
- `N` → 0

Target encoding here means converting the target labels to numeric class labels. It is not the same thing as target encoding a feature based on target statistics.

In [ ]:
feature_df["Loan_Status (Approved)"] = (
    feature_df["Loan_Status (Approved)"]
    .map({"Y": 1, "N": 0})
)

display(feature_df["Loan_Status (Approved)"].value_counts())

# 21. Transformation

Transformation changes the numerical representation of a feature.

Why transform?

A feature may be strongly skewed or have a scale/distribution that is inconvenient for some models.

Common transformations covered in the original teaching notebook include:

- Log transformation
- Square-root transformation
- Exponential transformation
- Box-Cox transformation
- Power transformations

**Do not use every transformation at the same time. Compare them and choose an appropriate representation.**

# 22. Log Transformation

Income and loan amounts can be right-skewed.

A safe choice when zeros may exist is:

**log1p(x) = log(1 + x)**

This is safer than `np.log(x)` when the feature can contain zero.

In [ ]:
transform_df = feature_df.copy()

for col in [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "TotalIncome"
]:
    transform_df[f"{col}_log"] = np.log1p(
        transform_df[col]
    )

display(transform_df[[
    "ApplicantIncome", "ApplicantIncome_log",
    "CoapplicantIncome", "CoapplicantIncome_log",
    "LoanAmount", "LoanAmount_log"
]].head())

## 23. Before vs After Log Transformation

The purpose is to reduce right-skewness and compress very large values.

The transformation does **not** magically make every distribution normal.
Always inspect the result.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["ApplicantIncome"], kde=True, ax=axes[0])
axes[0].set_title("ApplicantIncome — Original")

sns.histplot(transform_df["ApplicantIncome_log"], kde=True, ax=axes[1])
axes[1].set_title("ApplicantIncome — log1p")

plt.tight_layout()
plt.show()

# 24. Square-Root Transformation

Square-root transformation is another way to reduce right-skewness.

For non-negative values:

**sqrt(x)**

It is less aggressive than a logarithmic transformation.

In [ ]:
transform_df["LoanAmount_sqrt"] = np.sqrt(
    transform_df["LoanAmount"]
)

display(
    transform_df[
        ["LoanAmount", "LoanAmount_sqrt"]
    ].head()
)

# 25. Exponential Transformation — Teaching Demonstration

The original notebook includes exponential transformation.

Be careful: applying `exp(x)` to already large values can create extremely large numbers and overflow.

For the loan dataset, exponential transformation is **not recommended as the default preprocessing step**.

We demonstrate the concept on a small safe range instead.

In [ ]:
demo = pd.Series([0, 1, 2, 3], name="x")
demo_exp = np.exp(demo)

display(pd.DataFrame({
    "x": demo,
    "exp(x)": demo_exp
}))

# 26. Box-Cox Transformation

Box-Cox is designed for strictly positive values.

It searches for a power parameter λ that can make a distribution more symmetric.

For a feature containing zero, Box-Cox cannot be applied directly.

LoanAmount is suitable only after ensuring it is strictly positive.

In [ ]:
positive_loan = df["LoanAmount"].dropna()

# Verify the requirement.
print("Minimum LoanAmount:", positive_loan.min())

if (positive_loan > 0).all():
    loan_boxcox, lambda_value = boxcox(positive_loan)
    print("Estimated lambda:", lambda_value)

    plt.figure(figsize=(7, 4))
    sns.histplot(loan_boxcox, kde=True)
    plt.title("LoanAmount — Box-Cox transformed")
    plt.show()
else:
    print("Box-Cox cannot be applied directly because non-positive values exist.")

# 27. Yeo-Johnson Transformation

Yeo-Johnson is often a convenient power transformation because it can handle zero and negative values.

Use it when you want a learned power transformation without the strict-positive requirement of Box-Cox.

In [ ]:
yj = PowerTransformer(method="yeo-johnson")

loan_yj = yj.fit_transform(
    df[["LoanAmount"]].fillna(df["LoanAmount"].median())
)

plt.figure(figsize=(7, 4))
sns.histplot(loan_yj.ravel(), kde=True)
plt.title("LoanAmount — Yeo-Johnson transformed")
plt.show()

# 28. Compare Transformations

For a single feature, transformed versions are alternatives.

Do not automatically give the model all of these:

- LoanAmount
- LoanAmount_log
- LoanAmount_sqrt
- LoanAmount_boxcox
- LoanAmount_yj

Instead, compare their distributions and model performance, then choose an appropriate representation.

# 29. Feature Scaling

Features can have very different numerical ranges.

For example:

- ApplicantIncome can be in thousands
- LoanAmount can be hundreds
- Credit_History is 0/1

Scaling can be important for algorithms that depend on distance, magnitude, or gradient optimization.

Examples:

- KNN
- K-Means
- SVM
- Logistic Regression
- Neural Networks

Tree-based models such as Decision Trees and Random Forests generally do not require feature scaling for their split decisions.

# 30. Min-Max Scaling / Normalization

Formula:

**X_scaled = (X − X_min) / (X_max − X_min)**

Default output range:

**0 to 1**

MinMaxScaler is sensitive to outliers because the minimum and maximum determine the range.

In [ ]:
sample = df[[
    "ApplicantIncome",
    "LoanAmount"
]].copy()

sample = sample.fillna(sample.median())

mm = MinMaxScaler()

sample_minmax = pd.DataFrame(
    mm.fit_transform(sample),
    columns=sample.columns
)

display(sample_minmax.head())

# 31. Standardization

Standardization, also called Z-score scaling:

**z = (x − mean) / standard deviation**

After transformation, each feature is centered around approximately:

- Mean = 0
- Standard deviation = 1

StandardScaler is affected by outliers because it uses the mean and standard deviation.

In [ ]:
sample = df[[
    "ApplicantIncome",
    "LoanAmount"
]].copy()

sample = sample.fillna(sample.median())

std_scaler = StandardScaler()

sample_standard = pd.DataFrame(
    std_scaler.fit_transform(sample),
    columns=sample.columns
)

display(sample_standard.head())
print("\nMeans:")
print(sample_standard.mean())
print("\nStandard deviations:")
print(sample_standard.std(ddof=0))

# 32. Robust Scaling

RobustScaler uses statistics based on the median and interquartile range.

It is useful when features contain substantial outliers.

Conceptually:

**Robust scaled value = (x − median) / IQR**

This makes it less sensitive to extreme observations than StandardScaler.

In [ ]:
sample = df[[
    "ApplicantIncome",
    "LoanAmount"
]].copy()

sample = sample.fillna(sample.median())

robust = RobustScaler()

sample_robust = pd.DataFrame(
    robust.fit_transform(sample),
    columns=sample.columns
)

display(sample_robust.head())

# 33. Scaling — Important Rule

**Never fit preprocessing using the entire dataset before the train/test split.**

Wrong:

```python
scaler.fit_transform(X)
train_test_split(...)
```

The scaler has seen information from the future test set.

Correct:

```text
Split data
   ↓
Fit scaler on training data
   ↓
Transform training data
   ↓
Transform test data using the same fitted scaler
```

This prevents data leakage.

# 34. Final Recommended Approach — Pipeline

A robust ML workflow should put preprocessing inside a `Pipeline` and `ColumnTransformer`.

Benefits:

- Missing-value handling is reproducible
- Encoding is reproducible
- Scaling is reproducible
- Training statistics are learned only from training data
- Test data stays unseen during fitting
- The same preprocessing can be used for future predictions

In [ ]:
# Start from the raw data again for a clean, leakage-safe workflow.
data = pd.read_csv("loan_approved.csv").copy()

# Separate target.
X = data.drop(columns=["Loan_Status (Approved)", "Loan_ID"])
y = data["Loan_Status (Approved)"].map({"N": 0, "Y": 1})

# Convert Dependents to a numeric ordinal feature.
X["Dependents"] = X["Dependents"].replace({"3+": 3})
X["Dependents"] = pd.to_numeric(X["Dependents"], errors="coerce")

# Feature creation.
X["TotalIncome"] = X["ApplicantIncome"] + X["CoapplicantIncome"]

X["IncomePerDependent"] = (
    X["TotalIncome"] / (X["Dependents"] + 1)
)

X["LoanIncomeRatio"] = np.where(
    X["TotalIncome"] > 0,
    X["LoanAmount"] / X["TotalIncome"],
    np.nan
)

X["LoanToApplicantIncome"] = np.where(
    X["ApplicantIncome"] > 0,
    X["LoanAmount"] / X["ApplicantIncome"],
    np.nan
)

# We will use log-transformed income/loan features in place of their
# highly skewed originals for this demonstration.
for col in [
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "TotalIncome",
    "IncomePerDependent",
]:
    X[f"{col}_log"] = np.log1p(X[col])

# Drop the original highly skewed versions chosen for this demonstration.
X = X.drop(columns=[
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "TotalIncome",
    "IncomePerDependent"
])

print("Final feature columns:")
print(X.columns.tolist())

# 35. Define Columns for the Pipeline

After feature creation, the pipeline will:

### Numerical
- Impute missing values with median
- Standardize

### Categorical
- Impute missing values with most frequent category
- One-hot encode
- Ignore unseen categories at prediction time

In [ ]:
numeric_pipeline_features = [
    "Loan_Amount_Term",
    "Credit_History",
    "Dependents",
    "LoanIncomeRatio",
    "LoanToApplicantIncome",
    "ApplicantIncome_log",
    "CoapplicantIncome_log",
    "LoanAmount_log",
    "TotalIncome_log",
    "IncomePerDependent_log",
]

categorical_pipeline_features = [
    "Gender",
    "Married",
    "Education",
    "Self_Employed",
    "Property_Area"
]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_pipeline_features),
    ("cat", categorical_pipeline, categorical_pipeline_features)
])

# 36. Train/Test Split

We split before fitting the preprocessing steps.

`stratify=y` preserves approximately the same target-class proportion in train and test data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

# 37. Fit and Transform

The preprocessor learns parameters from `X_train` only.

Then:

- `fit_transform(X_train)` learns + transforms
- `transform(X_test)` only transforms using training parameters

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape :", X_test_processed.shape)

# 38. What Have We Done?

Our final feature-engineering workflow is:

```text
Raw Loan Data
     ↓
Remove Loan_ID
     ↓
Separate target
     ↓
Convert Dependents
     ↓
Create TotalIncome
     ↓
Create IncomePerDependent
     ↓
Create LoanIncomeRatio
     ↓
Create LoanToApplicantIncome
     ↓
Log-transform selected skewed features
     ↓
Train/Test Split
     ↓
Numerical Imputation + Standardization
     ↓
Categorical Imputation + One-Hot Encoding
     ↓
Model-ready matrix
```

# 39. Optional: Compare Scaling Techniques

For teaching, demonstrate that the scaler changes the representation, not the underlying observation.

Use:

- `StandardScaler()` → mean 0, standard deviation 1
- `MinMaxScaler()` → usually 0 to 1
- `RobustScaler()` → median/IQR based

Choose based on the algorithm, feature distribution, and outliers.

**Do not apply StandardScaler → MinMaxScaler → RobustScaler sequentially just because all three are available. They are alternative scaling strategies.**

# 40. Classroom Summary

### Feature Engineering means:

**Raw data → useful model features**

### For this loan dataset, we demonstrated:

✅ Data inspection  
✅ YData Profiling  
✅ Missing-value identification  
✅ Duplicate checking  
✅ Identifier removal  
✅ Numerical/categorical separation  
✅ IQR outlier detection  
✅ Outlier handling options  
✅ Feature creation  
✅ Binary encoding  
✅ Ordinal representation  
✅ One-hot encoding  
✅ Target encoding  
✅ Log transformation  
✅ Square-root transformation  
✅ Box-Cox  
✅ Yeo-Johnson  
✅ Min-Max scaling  
✅ Standardization  
✅ Robust scaling  
✅ Train/test split  
✅ Data-leakage prevention  
✅ Pipeline + ColumnTransformer  

### Final message for students

**Feature engineering is not “apply everything.”**

It is:

**Understand → Choose → Transform → Validate → Keep what helps.**

# 41. Student Practice

Try these exercises:

1. Find the percentage of missing values in every column.
2. Identify the columns containing potential outliers.
3. Compare `ApplicantIncome` before and after `log1p`.
4. Compare StandardScaler, MinMaxScaler and RobustScaler.
5. Explain why `Loan_ID` should not be used as a predictive feature.
6. Explain why `Property_Area` should be one-hot encoded.
7. Create a new feature called `LoanTermYears`.
8. Compare a model using the original features with a model using engineered features.
9. Explain data leakage with an example.
10. Explain why `np.log()` is unsafe for zero-valued income and why `np.log1p()` is safer.

# HAPPY LEARNING 🚀

**Remember: Good features + clean data + correct preprocessing = a strong foundation for Machine Learning.**